# MobileNetV2 + BiLSTM violence detection
Notebook PyTorch: cắt 16 frame liên tiếp từ mỗi video và lưu checkpoint `best.pt`.


In [ ]:
import os, random, json, cv2, numpy as np, pandas as pd, torch
import torch.nn as nn
from pathlib import Path
from PIL import Image
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import mobilenet_v2, MobileNet_V2_Weights

SEED=42; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DATA_ROOT=Path('/kaggle/input/datasets/bohong111/webcamtest/A-Dataset-for-Automatic-Violence-Detection-in-Videos-master/violence-detection-dataset')
# Nếu tên dataset trên Kaggle khác, chỉ sửa dòng DATA_ROOT này.
OUT_DIR=Path('/kaggle/working/mobilenetv2_bilstm'); OUT_DIR.mkdir(parents=True,exist_ok=True)
T=16; SIZE=224; BATCH=4; EPOCHS=20; VIDEO_EXTS={'.mp4','.avi','.mov','.mkv','.mpg','.mpeg','.webm'}
print(DEVICE, DATA_ROOT, DATA_ROOT.exists())

In [ ]:
def videos(folder):
    return sorted(p for p in Path(folder).rglob('*') if p.is_file() and p.suffix.lower() in VIDEO_EXTS)

nv=videos(DATA_ROOT/'non-violent'); vv=videos(DATA_ROOT/'violent')
print('NonViolence:',len(nv),'Violence:',len(vv))
assert nv and vv, 'Không tìm thấy video; kiểm tra DATA_ROOT.'
paths=[str(x) for x in nv+vv]; labels=[0]*len(nv)+[1]*len(vv)
train_p,temp_p,train_y,temp_y=train_test_split(paths,labels,test_size=.30,random_state=SEED,stratify=labels)
val_p,test_p,val_y,test_y=train_test_split(temp_p,temp_y,test_size=.50,random_state=SEED,stratify=temp_y)
print(len(train_p),len(val_p),len(test_p))

In [ ]:
mean=(.485,.456,.406); std=(.229,.224,.225)
nt=transforms.Compose([transforms.Resize((SIZE,SIZE)),transforms.ToTensor(),transforms.Normalize(mean,std)])
def read_frames(path):
    cap=cv2.VideoCapture(str(path)); n=int(cap.get(cv2.CAP_PROP_FRAME_COUNT)); out=[]
    if not cap.isOpened() or n<=0: cap.release(); return []
    for i in np.linspace(0,n-1,T).astype(int):
        cap.set(cv2.CAP_PROP_POS_FRAMES,int(i)); ok,bgr=cap.read()
        if ok: out.append(cv2.cvtColor(bgr,cv2.COLOR_BGR2RGB))
    cap.release()
    while len(out)<T: out.append(out[-1] if out else np.zeros((SIZE,SIZE,3),np.uint8))
    return out[:T]
class VideoDataset(Dataset):
    def __init__(self,p,y,train=False): self.p,self.y,self.train=p,y,train
    def __len__(self): return len(self.p)
    def __getitem__(self,i):
        fs=read_frames(self.p[i]); x=torch.stack([nt(Image.fromarray(f)) for f in fs])
        return x,int(self.y[i])
def loader(p,y,train=False): return DataLoader(VideoDataset(p,y,train),batch_size=BATCH,shuffle=train,num_workers=2,pin_memory=True)
train_loader=loader(train_p,train_y,True); val_loader=loader(val_p,val_y); test_loader=loader(test_p,test_y)

In [ ]:
class MobileNetBiLSTM(nn.Module):
    def __init__(self):
        super().__init__()
        net=mobilenet_v2(weights=MobileNet_V2_Weights.DEFAULT)
        self.cnn=net.features; self.pool=nn.AdaptiveAvgPool2d(1)
        self.lstm=nn.LSTM(1280,256,num_layers=2,batch_first=True,bidirectional=True,dropout=.4)
        self.head=nn.Sequential(nn.Dropout(.5),nn.Linear(512,2))
    def forward(self,x):
        b,t,c,h,w=x.shape; z=self.cnn(x.reshape(b*t,c,h,w)); z=self.pool(z).flatten(1).view(b,t,-1)
        z,_=self.lstm(z); return self.head(z[:,-1])
model=MobileNetBiLSTM().to(DEVICE); print('trainable:',sum(p.numel() for p in model.parameters() if p.requires_grad))

In [ ]:
opt=torch.optim.AdamW(model.parameters(),lr=1e-4,weight_decay=1e-4)
loss_fn=nn.CrossEntropyLoss(); scaler=torch.cuda.amp.GradScaler(enabled=DEVICE.type=='cuda')
def epoch(dl,train=False):
    model.train(train); ys=[]; ps=[]; ls=[]
    for x,y in tqdm(dl,leave=False):
        x=x.to(DEVICE,non_blocking=True); y=torch.tensor(y,device=DEVICE)
        if train: opt.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=DEVICE.type=='cuda'):
            logits=model(x); loss=loss_fn(logits,y)
        if train: scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        ls.append(loss.item()); ys+=y.cpu().tolist(); ps+=logits.argmax(1).detach().cpu().tolist()
    return float(np.mean(ls)),accuracy_score(ys,ps),f1_score(ys,ps,average='macro')
best=-1
for e in range(1,EPOCHS+1):
    tr=epoch(train_loader,True); va=epoch(val_loader)
    print(f'Epoch {e}: train loss {tr[0]:.4f} acc {tr[1]:.3f} f1 {tr[2]:.3f} | val loss {va[0]:.4f} acc {va[1]:.3f} f1 {va[2]:.3f}')
    if va[2]>best:
        best=va[2]; torch.save({'model':model.state_dict(),'cfg':{'data':{'classes':['non_violence','violence'],'num_frames':T,'image_size':SIZE},'model':{'backbone':'mobilenet_v2','pretrained':False,'freeze_backbone':False,'lstm_hidden':256,'lstm_layers':2,'bidirectional':True,'dropout':.5,'use_attention':False,'num_classes':2}},'val':va},OUT_DIR/'best.pt')
print('Saved:',OUT_DIR/'best.pt'); print('Test:',epoch(test_loader))